# Notebook 02 — Thu thập và làm sạch dữ liệu

Pipeline làm sạch theo `src.cleaner.clean_dataframe()`:

## 1. Thống kê trước khi làm sạch

In [ ]:
import pandas as pd
from src.data_manager import PropertyDataManager

mgr = PropertyDataManager('data/raw/real_estate_with_price_per_m2.csv')
raw = mgr.load_raw()
print('Trước làm sạch:')
print(f'  Số dòng: {len(raw)}')
print(f'  district unique: {raw["district"].nunique()} giá trị — gồm cả số "1" và tên "Bình Thạnh"')
print(f'  house_direction unique: {raw["house_direction"].nunique()} giá trị — gồm cả dạng ghép')
print(f'  area_m2: min={raw["area_m2"].min()}, max={raw["area_m2"].max()}')
print(f'  total_price: min={raw["total_price"].min():.0f}, max={raw["total_price"].max():.0f}')

## 2. Chuẩn hóa district (số → "Quận X")

In [ ]:
from src.cleaner import normalize_district
raw['district_clean'] = raw['district'].apply(normalize_district)
print('Trước — 10 giá trị district đầu:')
print(raw['district'].head(10).tolist())
print('\nSau — 10 district_clean đầu:')
print(raw['district_clean'].head(10).tolist())

## 3. Chuẩn hóa hướng nhà

In [ ]:
from src.cleaner import normalize_direction
raw['direction_clean'] = raw['house_direction'].apply(normalize_direction)
print('Các hướng chuẩn sau khi xử lý:')
print(raw['direction_clean'].value_counts(dropna=False))

## 4. Lọc outlier và tính lại price_per_m2

In [ ]:
cleaned, log, errors = mgr.clean()
print(f'Sau làm sạch: {len(cleaned)} dòng (bỏ {len(raw) - len(cleaned)} outlier)')
print(f'Cleaning log: {len(log)} dòng được ghi nhận')
print(f'\nPhân bố issue_type trong log:')
print(log['issue_type'].value_counts())
print(f'\nErrors: {errors}')

## 5. Merge với nguồn 2 — tiện ích phường/xã

In [ ]:
amenities = pd.read_csv('data/raw/neighborhood_amenities.csv')
merged = mgr.merge_amenities(cleaned, amenities)
print(f'Merged shape: {merged.shape}')
print(f'Tin khớp amenity: {merged["amenity_score"].notna().sum()} / {len(merged)}')

## 6. Lưu kết quả làm sạch

In [ ]:
from pathlib import Path
mgr.save_cleaned(
    cleaned, log,
    cleaned_path=Path('data/processed/listings_clean.csv'),
    log_path=Path('data/logs/cleaning_log.csv'),
    error_path=Path('data/logs/error_log.txt'),
    errors=errors,
)
merged.to_csv('data/processed/listings_with_amenities.csv', index=False)
print('Saved → data/processed/ và data/logs/')

## 7. Kết luận

- Loại **32 dòng outlier** (chủ yếu giá < 100 triệu hoặc diện tích > 1000m²).
- Tất cả `district` đã chuẩn — 22 giá trị duy nhất ở `district_clean`.
- `house_direction` đã chuẩn — 8 giá trị chính + missing.
- 1987/2968 dòng khớp với dữ liệu tiện ích phường (≈67%); phần còn lại là phường chưa có trong `neighborhood_amenities.csv` (do tập phụ chỉ 100 dòng).

Tiếp theo: mở `03_eda.ipynb` để xem EDA.